# Clean and prepare fact-checking datasets

This notebook loads the raw fact-news cache and TurnBackHoax data, removes unwanted titles and duplicates, normalizes text, assigns labels, samples the fact dataset, and saves cleaned CSV files.

Labels: `0` = fact/news and `1` = fake news. Normalization removes punctuation and emoji, lowercases text, and collapses repeated whitespace while preserving word boundaries.

In [1]:
# Install once if needed, then restart the kernel.
# %pip install pandas numpy sentence-transformers

In [2]:
import csv
import random
import re
import unicodedata
from collections import defaultdict
from datetime import date, datetime, timedelta
from datetime import timedelta

import pandas as pd
from sentence_transformers import SentenceTransformer
from pathlib import Path

SEED = 42
TARGET_FACT_ROWS = 6394
SEM_THRESHOLD = 0.78
SEM_WINDOW_DAYS = 3
EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# This works when the notebook is opened from the repository root or preprocessing/.
ROOT = Path.cwd()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent
RAW_DIR = ROOT / "datasets" / "raw"
OUTPUT_DIR = ROOT / "datasets" / "cleaned"
FACT_CACHE = RAW_DIR / "raw_fact_cache.jsonl"
FAKE_CSV = RAW_DIR / "turnbackhoax_2024_2026_raw.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {ROOT}")
print(f"Fact cache: {FACT_CACHE}")
print(f"Fake-news CSV: {FAKE_CSV}")

c:\Users\screm\anaconda3\envs\peft\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: c:\Users\screm\Code\Thesis\Code
Fact cache: c:\Users\screm\Code\Thesis\Code\datasets\raw\raw_fact_cache.jsonl
Fake-news CSV: c:\Users\screm\Code\Thesis\Code\datasets\raw\turnbackhoax_2024_2026_raw.csv


## Load raw data

In [3]:
import json

def load_fact_cache(path):
    rows = []
    with path.open(encoding="utf-8") as file:
        for line in file:
            record = json.loads(line)
            for title in record.get("titles", []):
                rows.append({
                    "title": title,
                    "date": record.get("date"),
                    "source": record.get("source"),
                    "label": 0,
                })
    return pd.DataFrame(rows)


def load_fake_news(path):
    frame = pd.read_csv(path)
    frame = frame.rename(columns={"berita": "title", "tanggal_publikasi": "date"})
    frame["label"] = 1
    frame["source"] = "turnbackhoax"
    return frame[["title", "date", "source", "label"]]

facts_raw = load_fact_cache(FACT_CACHE)
fake_raw = load_fake_news(FAKE_CSV)
print(f"Raw fact titles: {len(facts_raw):,}")
print(f"Raw fake-news rows: {len(fake_raw):,}")
display(facts_raw.head())
display(fake_raw.head())

Raw fact titles: 11,951
Raw fake-news rows: 6,391


,title,date,source,label
0,"Tahun Baru, Warga Padati Sejumlah Tempat Wisat...",2024-01-01,detik,0
1,Kala Ganjar Tanya Influencer: Mau Makan Siang ...,2024-01-01,detik,0
2,Selamat Tahun Baru 2024!,2024-01-01,detik,0
3,"Tahun Baru 2024, Begini Penampakan Atraksi 500...",2024-01-01,detik,0
4,"Tinjau 3 Stasiun, Menhub: Kereta Api Sangat Di...",2024-01-01,kompas,0


,title,date,source,label
0,[SALAH] Purbaya Diangkat Jadi Kepala Bea Cukai,23/09/2026,turnbackhoax,1
1,[SALAH] Panas El Nino Berkaitan dengan Gelomba...,23/09/2026,turnbackhoax,1
2,[SALAH] World Economic Forum (WEF) Mau Campurk...,22/09/2026,turnbackhoax,1
3,[PENIPUAN] Dana Bantuan dari KemenP2MI untuk K...,22/09/2026,turnbackhoax,1
4,[PENIPUAN] Tautan Pendaftaran Lowongan Cimory,22/09/2026,turnbackhoax,1


## Clean and normalize titles

The exclusion terms are matched case-insensitively. `cara` is treated as a whole word, so words such as `secara` are not accidentally removed.

In [4]:
EXCLUDED_PATTERNS = [
    re.compile(r"\[hoaks\]", re.IGNORECASE),
    re.compile(r"lirik lagu", re.IGNORECASE),
    re.compile(r"lirik dan chord", re.IGNORECASE),
    re.compile(r"resep", re.IGNORECASE),
    re.compile(r"\bcara\b", re.IGNORECASE),
    re.compile(r"\.\.\."),
]
BRACKETED_LABEL_PATTERN = re.compile(r"\[[^\]]*\]\s*", re.IGNORECASE)


def is_excluded(title):
    return any(pattern.search(str(title)) for pattern in EXCLUDED_PATTERNS)


def normalize_text(title):
    text = unicodedata.normalize("NFKC", str(title)).lower()
    # Replace punctuation, symbols, and emoji with spaces, retaining word boundaries.
    text = "".join(character if character.isalnum() or character.isspace() else " "
                    for character in text)
    return re.sub(r"\s+", " ", text).strip()


def prepare_frame(frame, remove_bracketed_labels=False):
    prepared = frame.copy()
    prepared = prepared.dropna(subset=["title"])
    prepared["title"] = prepared["title"].astype(str).str.strip()
    prepared = prepared[prepared["title"].str.len() > 0]
    prepared = prepared[~prepared["title"].map(is_excluded)]
    if remove_bracketed_labels:
        prepared["title"] = prepared["title"].str.replace(
            BRACKETED_LABEL_PATTERN,
            "",
            regex=True,
        ).str.strip()
        prepared = prepared[prepared["title"].str.len() > 0]
    prepared["normalized_title"] = prepared["title"].map(normalize_text)
    prepared = prepared[prepared["normalized_title"].str.len() > 0]
    prepared = prepared.drop_duplicates(subset=["normalized_title"], keep="first")
    prepared["date"] = pd.to_datetime(prepared["date"], dayfirst=True, errors="coerce")
    return prepared.reset_index(drop=True)


facts = prepare_frame(facts_raw)
fake_news = prepare_frame(fake_raw, remove_bracketed_labels=True)
print(f"Facts after filtering and exact deduplication: {len(facts):,}")
print(f"Fake news after filtering, label cleanup, and exact deduplication: {len(fake_news):,}")

Facts after filtering and exact deduplication: 10,818
Fake news after filtering, label cleanup, and exact deduplication: 6,318


## Semantic deduplication

Titles with cosine similarity at least `0.78` are considered the same story when their dates are within three days. The fact dataset follows the same date-window approach as `combined_fact_scraper.py`; the same rule is also applied to fake-news rows.

In [ ]:
model = SentenceTransformer(EMB_MODEL)


def semantic_deduplicate(frame):
    if frame.empty:
        return frame.copy()
    working = frame.reset_index(drop=True).copy()
    embeddings = model.encode(
        working["title"].tolist(),
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True,
    )
    order = list(range(len(working)))
    random.Random(SEED).shuffle(order)
    kept = []
    duplicates = []
    by_date = defaultdict(list)
    for index in order:
        current_date = working.at[index, "date"]
        candidates = []
        if pd.notna(current_date):
            current_date = current_date.date()
            for offset in range(-SEM_WINDOW_DAYS, SEM_WINDOW_DAYS + 1):
                candidates.extend(by_date.get(current_date + timedelta(days=offset), []))
        if candidates:
            similarities = embeddings[candidates] @ embeddings[index]
            best_position = int(similarities.argmax())
            best_similarity = float(similarities[best_position])
            if best_similarity >= SEM_THRESHOLD:
                kept_index = candidates[best_position]
                duplicates.append({
                    "duplicate_title": working.at[index, "title"],
                    "kept_title": working.at[kept_index, "title"],
                    "date": working.at[index, "date"],
                    "similarity": best_similarity,
                })
                continue
        kept.append(index)
        if pd.notna(working.at[index, "date"]):
            by_date.setdefault(working.at[index, "date"].date(), []).append(index)
    duplicate_report = pd.DataFrame(duplicates)
    print(f"Semantic duplicates removed: {len(duplicate_report):,}")
    if not duplicate_report.empty:
        display(duplicate_report.sort_values("similarity", ascending=False).reset_index(drop=True))
    return working.iloc[kept].reset_index(drop=True)


facts = semantic_deduplicate(facts)
fake_news = semantic_deduplicate(fake_news)
print(f"Facts after semantic deduplication: {len(facts):,}")
print(f"Fake news after semantic deduplication: {len(fake_news):,}")

Batches:   4%|▎         | 6/170 [00:07<02:39,  1.03it/s]

## Sample facts to match the fake-news dataset

The fact dataset is sampled to the same number of rows as the deduplicated fake-news dataset. Sampling is balanced by source and month; rows with missing dates are included in an `unknown` month group.

In [ ]:
def sample_facts(frame, target):
    randomizer = random.Random(SEED)
    sources = list(frame["source"].dropna().unique())
    base, extra = divmod(target, len(sources)) if sources else (0, 0)
    selected = []
    for source_index, source in enumerate(sources):
        source_target = base + (1 if source_index < extra else 0)
        source_rows = frame[frame["source"] == source].copy()
        source_rows["month"] = source_rows["date"].dt.strftime("%Y-%m").fillna("unknown")
        months = {month: group.index.tolist() for month, group in source_rows.groupby("month")}
        for indices in months.values():
            randomizer.shuffle(indices)
        picked = []
        for month in sorted(months):
            if len(picked) >= source_target:
                break
            if months[month]:
                picked.append(months[month].pop())
        while len(picked) < source_target and any(months.values()):
            for month in sorted(months):
                if months[month] and len(picked) < source_target:
                    picked.append(months[month].pop())
        selected.extend(picked)
        print(f"{source}: {len(picked):,}/{source_target:,}")
    return frame.loc[selected].copy()

facts = sample_facts(facts, target=len(fake_news))
print(f"Selected fact rows: {len(facts):,}")
print(f"Fake-news rows for matching: {len(fake_news):,}")

kompas: 2,021/2,021
liputan6: 2,021/2,021
detik: 2,020/2,020
Selected fact rows: 6,062
Fake-news rows for matching: 6,062


## Save cleaned datasets

In [ ]:
def output_frame(frame):
    result = frame[["normalized_title", "label"]].rename(columns={"normalized_title": "berita"})
    return result.sort_values(["label", "berita"]).reset_index(drop=True)

facts_output = output_frame(facts)
fake_output = output_frame(fake_news)
combined_output = pd.concat([facts_output, fake_output], ignore_index=True)
combined_output = combined_output.sample(frac=1, random_state=SEED).reset_index(drop=True)

facts_output.to_csv(OUTPUT_DIR / "fact_news_cleaned.csv", index=False, encoding="utf-8")
fake_output.to_csv(OUTPUT_DIR / "fake_news_cleaned.csv", index=False, encoding="utf-8")
combined_output.to_csv(OUTPUT_DIR / "combined_news_cleaned.csv", index=False, encoding="utf-8")

print(f"Saved: {OUTPUT_DIR / 'fact_news_cleaned.csv'} ({len(facts_output):,} rows)")
print(f"Saved: {OUTPUT_DIR / 'fake_news_cleaned.csv'} ({len(fake_output):,} rows)")
print(f"Saved: {OUTPUT_DIR / 'combined_news_cleaned.csv'} ({len(combined_output):,} rows)")
display(combined_output.head())

Saved: c:\Users\screm\Code\Thesis\Code\datasets\cleaned\fact_news_cleaned.csv (6,062 rows)
Saved: c:\Users\screm\Code\Thesis\Code\datasets\cleaned\fake_news_cleaned.csv (6,062 rows)
Saved: c:\Users\screm\Code\Thesis\Code\datasets\cleaned\combined_news_cleaned.csv (12,124 rows)


,berita,label
0,sidak di pasar jaya lenteng agung mentan amran...,0
1,jawaban soal berikut prinsip prinsip demokrasi...,0
2,anak krakatau erupsi penyeberangan bakauheni m...,0
3,akun whatsapp kajari halmahera barat kusuma ja...,1
4,ketua mk ai pada sistem peradilan tingkatkan e...,0


## Create train, validation, and test splits

The full all-years combined dataset is split into three configurations: 80/10/10, 70/10/20, and 70/15/15. Only the 70/10/20 configuration gets stratified 5%, 10%, and 50% training subsets. The source row ID is used to verify that train, validation, and test are disjoint.

In [ ]:
from sklearn.model_selection import train_test_split


def split_dataset(dataset, train_fraction, validation_fraction, test_fraction):
    if not abs(train_fraction + validation_fraction + test_fraction - 1.0) < 1e-9:
        raise ValueError("Split fractions must add up to 1.0.")

    train, remaining = train_test_split(
        dataset,
        train_size=train_fraction,
        stratify=dataset["label"],
        random_state=SEED,
    )
    validation_fraction_of_remaining = validation_fraction / (validation_fraction + test_fraction)
    validation, test = train_test_split(
        remaining,
        train_size=validation_fraction_of_remaining,
        stratify=remaining["label"],
        random_state=SEED,
    )
    return tuple(
        split.reset_index(drop=True)
        for split in (train, validation, test)
    )


def stratified_subset(dataset, fraction):
    return dataset.groupby("label", group_keys=False).sample(
        frac=fraction,
        random_state=SEED,
    ).sample(frac=1, random_state=SEED).reset_index(drop=True)


def assert_disjoint_splits(train, validation, test):
    split_ids = {
        "train": set(train["_row_id"]),
        "validation": set(validation["_row_id"]),
        "test": set(test["_row_id"]),
    }
    for first_name, second_name in (("train", "validation"), ("train", "test"), ("validation", "test")):
        overlap = split_ids[first_name] & split_ids[second_name]
        if overlap:
            raise AssertionError(f"{first_name} and {second_name} overlap on {len(overlap)} rows.")


# Use the full all-years dataset for the training experiments.
dataset_for_splitting = combined_output.copy()
dataset_for_splitting.insert(0, "_row_id", range(len(dataset_for_splitting)))
DATASET_NAME = "split"

split_experiments = {
    "80_10_10": (0.80, 0.10, 0.10),
    "70_10_20": (0.70, 0.10, 0.20),
    "70_15_15": (0.70, 0.15, 0.15),
}

SPLIT_OUTPUT_DIR = OUTPUT_DIR / "splits"
SPLIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for experiment_name, fractions in split_experiments.items():
    train, validation, test = split_dataset(dataset_for_splitting, *fractions)
    assert_disjoint_splits(train, validation, test)
    experiment_dir = SPLIT_OUTPUT_DIR / f"{DATASET_NAME}_{experiment_name}"
    experiment_dir.mkdir(parents=True, exist_ok=True)

    output_columns = ["berita", "label"]
    train[output_columns].to_csv(experiment_dir / "train.csv", index=False, encoding="utf-8")
    if experiment_name == "70_10_20":
        for subset_fraction in (0.05, 0.10, 0.50):
            subset = stratified_subset(train, subset_fraction)
            subset_name = f"train_{int(subset_fraction * 100)}pct.csv"
            subset[output_columns].to_csv(experiment_dir / subset_name, index=False, encoding="utf-8")
    validation[output_columns].to_csv(experiment_dir / "val.csv", index=False, encoding="utf-8")
    test[output_columns].to_csv(experiment_dir / "test.csv", index=False, encoding="utf-8")

    print(f"{DATASET_NAME} {experiment_name}: train={len(train):,}, val={len(validation):,}, test={len(test):,}")
    print("  train labels:", train["label"].value_counts().sort_index().to_dict())
    print("  val labels:", validation["label"].value_counts().sort_index().to_dict())
    print("  test labels:", test["label"].value_counts().sort_index().to_dict())

split 80_10_10: train=9,699, val=1,212, test=1,213
  train labels: {0: 4849, 1: 4850}
  val labels: {0: 606, 1: 606}
  test labels: {0: 607, 1: 606}
split 70_10_20: train=8,486, val=1,212, test=2,426
  train labels: {0: 4243, 1: 4243}
  val labels: {0: 606, 1: 606}
  test labels: {0: 1213, 1: 1213}
split 70_15_15: train=8,486, val=1,819, test=1,819
  train labels: {0: 4243, 1: 4243}
  val labels: {0: 909, 1: 910}
  test labels: {0: 910, 1: 909}
